In [12]:
#gen ESM-2 embeddings for training sequences
import torch
import esm
import pandas as pd
import numpy as np
import json

# load device GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.10.0


In [7]:
df = pd.read_csv("card_sequences.csv")

In [8]:
#load esm-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.to(device)
model.eval()

# prep model for inference
last_layer = len(model.layers)
batch_converter = alphabet.get_batch_converter()  # convert seqs to tokens

In [9]:
def tokenize_batch(sequences):
    # convert a batch of sequences to tokens (MUCH faster than one-at-a-time)
    batch = [(f"orf{i}", seq) for i, seq in enumerate(sequences)]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch)
    return batch_tokens.to(device=device, dtype=torch.long)

In [10]:
def extract_embeddings_batch(sequences):
    # get sequence embeddings from final layer for a batch
    tokens = tokenize_batch(sequences)

    with torch.no_grad():
        output = model(tokens, repr_layers=[last_layer], return_contacts=False)

    full_embedding = output["representations"][last_layer]  # (B, L_tokens, d)

    pad_idx = alphabet.padding_idx

    # mean embedding, average over real tokens only (ignore padding)
    mask = (tokens != pad_idx)[:, 1:-1]                # (B, L-2), True for real tokens
    reps_r = full_embedding[:, 1:-1, :]                # (B, L-2, d)

    denom = mask.sum(dim=1).clamp(min=1).unsqueeze(-1) # (B,1)
    mean_embedding = (reps_r * mask.unsqueeze(-1)).sum(dim=1) / denom

    # cls embedding, first token in sequence
    cls_embedding = full_embedding[:, 0, :]  # (B, d)

    return (
        mean_embedding.cpu().numpy(),
        cls_embedding.cpu().numpy()
    )

In [ ]:
#run emb extraction for all sequences and save to new dataframe, allow 10-20 minutes for inference
emb_df = df.copy()

for _,row in df.iterrows():
    seq = row["Sequence"]
    mean_emb, cls_emb = extract_embeddings_batch([seq])
    emb_df.at[_, "MeanEmbedding"] = json.dumps(mean_emb.tolist())
    emb_df.at[_, "CLSEmbedding"] = json.dumps(cls_emb.tolist())

#save to csv
emb_df["EmbeddingDim"] = len(mean_emb[0])
emb_df.to_csv("embeddings.csv", index=False)
emb_df.head()

,Family,Gene,Sequence,Carbapenemase,MeanEmbedding,CLSEmbedding,EmbeddingDim
0,SHV,SHV-52,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,0,"[[0.05962217226624489, -0.07687947154045105, 0...","[[0.07300693541765213, -0.0206560418009758, 0....",1280
1,CTX-M,CTX-M-130,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGG...,0,"[[0.05877629667520523, -0.06891713291406631, -...","[[0.10748688131570816, -0.044926486909389496, ...",1280
2,TEM,TEM-126,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...,0,"[[0.059695929288864136, -0.05530138313770294, ...","[[0.07826980203390121, -0.01159658282995224, 0...",1280
3,TEM,TEM-72,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDKLGARVGYIE...,0,"[[0.056167904287576675, -0.04905174672603607, ...","[[0.07821495085954666, -0.010604368522763252, ...",1280
4,TEM,TEM-59,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDKLGARVGYIE...,0,"[[0.07185406237840652, -0.056960783898830414, ...","[[0.08561597019433975, -0.01823832094669342, 0...",1280
